In [2]:
# ==============================================================================
# Complete Self-Contained Pipeline (Pure Python - No Cell Magics Needed)
# ==============================================================================

import os
import sys
from dataclasses import dataclass
from typing import Optional, Dict, Any
import pandas as pd

# ------------------------------------------------------------------------------
# 1. Create File Structure via Standard Python File I/O
# ------------------------------------------------------------------------------
os.makedirs("src", exist_ok=True)

# Write src/__init__.py
with open("src/__init__.py", "w") as f:
    f.write("# Module marker\n")

# Write src/data_types.py
with open("src/data_types.py", "w") as f:
    f.write('''from dataclasses import dataclass
from typing import Optional

@dataclass
class RegionFeatures:
    """
    Standardized, dataset-agnostic intermediate representation for spatial regions.
    Decoupled from specific dataset formats (e.g., SemanticKITTI, nuScenes, Kaggle).
    """
    region_id: int
    x: float
    y: float
    distance: float
    terrain_complexity: float
    point_count: int
    semantic_label: Optional[str]
    semantic_importance: float
    confidence: float
    dynamic_relevance: float
    uncertainty: float
''')

# Write src/interface_validator.py
with open("src/interface_validator.py", "w") as f:
    f.write('''from src.data_types import RegionFeatures

def validate_region_features(region: RegionFeatures) -> bool:
    """
    Validates boundary conditions, value constraints, and data types of a RegionFeatures instance.
    Raises ValueError or TypeError with explicit diagnostic messages on violations.
    """
    if not isinstance(region.region_id, int) or isinstance(region.region_id, bool):
        raise TypeError(f"region_id must be an integer, got {type(region.region_id).__name__} (value: {region.region_id})")

    if not isinstance(region.x, (float, int)) or not isinstance(region.y, (float, int)):
        raise TypeError(f"Coordinates (x, y) must be numeric floats. Got x={type(region.x).__name__}, y={type(region.y).__name__}")

    # Ensure canonical float type representation
    if not isinstance(region.x, float) or not isinstance(region.y, float):
        raise TypeError(f"Coordinates (x, y) must strictly be floats. Found x:{type(region.x)}, y:{type(region.y)}")

    if not isinstance(region.point_count, int) or isinstance(region.point_count, bool) or region.point_count < 0:
        raise ValueError(f"point_count must be a non-negative integer, got {region.point_count}")

    if region.distance < 0.0:
        raise ValueError(f"distance must be non-negative (>= 0.0), got {region.distance}")

    unit_interval_fields = {
        "semantic_importance": region.semantic_importance,
        "confidence": region.confidence,
        "dynamic_relevance": region.dynamic_relevance,
        "uncertainty": region.uncertainty,
        "terrain_complexity": region.terrain_complexity,
    }

    for name, val in unit_interval_fields.items():
        if not isinstance(val, (float, int)):
            raise TypeError(f"Field '{name}' must be a float in [0.0, 1.0], got {type(val).__name__}")
        if not (0.0 <= float(val) <= 1.0):
            raise ValueError(f"Field '{name}' must be bounded in [0.0, 1.0], got {val}")

    return True
''')

# Ensure working directory is in sys.path so modules can be imported
if "." not in sys.path:
    sys.path.insert(0, ".")

# ------------------------------------------------------------------------------
# 2. Import from newly generated modules
# ------------------------------------------------------------------------------
from src.data_types import RegionFeatures
from src.interface_validator import validate_region_features

# ------------------------------------------------------------------------------
# 3. Pipeline Stages: Mocks & Explanations
# ------------------------------------------------------------------------------
# NOTE ON CONFIDENCE VS. UNCERTAINTY:
# - Confidence: Model belief in its prediction (e.g. classification softmax score).
# - Uncertainty: Epistemic/aleatoric ambiguity or spatial/sensor noise (e.g. LiDAR range noise, occlusion).
# High confidence can coexist with high uncertainty (e.g. distant or occluded target).

class MockImportanceEngine:
    """
    TODO: Replace this mock with the real Stage 1 Importance Engine.
    Computes a normalized importance score in [0.0, 1.0] from RegionFeatures.
    """
    def __init__(self):
        self.w_semantic = 0.35
        self.w_dynamic = 0.30
        self.w_terrain = 0.15
        self.w_uncertainty = 0.10  # High uncertainty warrants map attention
        self.w_confidence = 0.05
        self.w_distance = 0.05    # Closer objects get slight baseline priority

    def compute_importance(self, region: RegionFeatures) -> float:
        max_range = 100.0
        distance_factor = max(0.0, 1.0 - min(region.distance, max_range) / max_range)

        raw_score = (
            self.w_semantic * region.semantic_importance
            + self.w_dynamic * region.dynamic_relevance
            + self.w_terrain * region.terrain_complexity
            + self.w_uncertainty * region.uncertainty
            + self.w_confidence * region.confidence
            + self.w_distance * distance_factor
        )
        return float(min(1.0, max(0.0, raw_score)))


class MockResolutionEngine:
    """
    TODO: Replace this mock with the real Stage 1 Resolution Engine.
    Maps an importance score in [0.0, 1.0] to a grid resolution (cell size in meters).
    Higher importance -> finer resolution (smaller cell size).
    """
    def compute_resolution(self, importance_score: float) -> float:
        if importance_score >= 0.70:
            return 0.10  # Fine grid (0.10m / 10cm cells)
        elif importance_score >= 0.40:
            return 0.50  # Medium grid (0.50m / 50cm cells)
        else:
            return 2.00  # Coarse grid (2.00m / 200cm cells)


class MockAdaptive25DMapper:
    """
    TODO: Replace this mock with the real Stage 1 2.5D Mapping Module.
    Accepts region properties and chosen resolution, producing a 2.5D cell record.
    """
    def generate_cell_record(self, region: RegionFeatures, resolution_m: float) -> Dict[str, Any]:
        return {
            "region_id": region.region_id,
            "anchor": (round(region.x, 2), round(region.y, 2)),
            "label": region.semantic_label,
            "resolution_m": resolution_m,
            "point_density": round(region.point_count / (resolution_m ** 2), 2),
            "elevation_layer": "2.5D_surface_height_estimated",
        }

# ------------------------------------------------------------------------------
# 4. Synthetic Test Dataset
# ------------------------------------------------------------------------------
synthetic_regions = [
    # 1. Distant Pedestrian (~70m): Safety-critical, dynamic, highly confident
    RegionFeatures(
        region_id=101,
        x=50.0,
        y=49.0,
        distance=70.0,
        terrain_complexity=0.20,
        point_count=45,
        semantic_label="Pedestrian",
        semantic_importance=0.95,
        confidence=0.88,
        dynamic_relevance=0.95,
        uncertainty=0.12,  # Low uncertainty, high confidence
    ),
    # 2. Road segment (~70m): Static background, low semantic importance
    RegionFeatures(
        region_id=102,
        x=45.0,
        y=53.7,
        distance=70.0,
        terrain_complexity=0.05,
        point_count=320,
        semantic_label="Road",
        semantic_importance=0.10,
        confidence=0.96,
        dynamic_relevance=0.02,
        uncertainty=0.04,  # High confidence, very low uncertainty
    ),
    # 3. Nearby Moving Car (~20m): High dynamic priority, fine resolution
    RegionFeatures(
        region_id=103,
        x=12.0,
        y=16.0,
        distance=20.0,
        terrain_complexity=0.15,
        point_count=480,
        semantic_label="Car",
        semantic_importance=0.85,
        confidence=0.92,
        dynamic_relevance=0.90,
        uncertainty=0.10,
    ),
    # 4. Dense Vegetation/Bush (~15m): High terrain roughness, high uncertainty
    RegionFeatures(
        region_id=104,
        x=10.0,
        y=-11.2,
        distance=15.0,
        terrain_complexity=0.85,
        point_count=210,
        semantic_label="Vegetation",
        semantic_importance=0.35,
        confidence=0.70,
        dynamic_relevance=0.10,
        uncertainty=0.75,  # High uncertainty due to porous structure
    ),
    # 5. Distant Building Facade (~95m): Static, planar, coarse candidate
    RegionFeatures(
        region_id=105,
        x=75.0,
        y=58.3,
        distance=95.0,
        terrain_complexity=0.10,
        point_count=130,
        semantic_label="Building",
        semantic_importance=0.30,
        confidence=0.85,
        dynamic_relevance=0.00,
        uncertainty=0.25,
    ),
]

# ------------------------------------------------------------------------------
# 5. Pipeline Execution
# ------------------------------------------------------------------------------
importance_engine = MockImportanceEngine()
resolution_engine = MockResolutionEngine()
mapper_25d = MockAdaptive25DMapper()

pipeline_results = []
pedestrian_res = None
road_res = None
pedestrian_imp = None
road_imp = None

for region in synthetic_regions:
    # 1. Interface validation
    validate_region_features(region)

    # 2. Importance estimation
    importance_score = importance_engine.compute_importance(region)

    # 3. Resolution allocation
    resolution_m = resolution_engine.compute_resolution(importance_score)

    # 4. 2.5D Mapping
    cell_record = mapper_25d.generate_cell_record(region, resolution_m)

    # Track metrics for validation assertions
    if region.semantic_label == "Pedestrian":
        pedestrian_imp = importance_score
        pedestrian_res = resolution_m
    elif region.semantic_label == "Road":
        road_imp = importance_score
        road_res = resolution_m

    pipeline_results.append({
        "region_id": region.region_id,
        "semantic_label": region.semantic_label,
        "distance": f"{region.distance:.1f}m",
        "importance_score": round(importance_score, 4),
        "resolution": f"{resolution_m:.2f}m",
        "confidence": region.confidence,
        "uncertainty": region.uncertainty,
    })

# Format and print table
df_summary = pd.DataFrame(pipeline_results)
cols_display = ["region_id", "semantic_label", "distance", "importance_score", "resolution", "confidence", "uncertainty"]
print("\n=== ADAPTIVE 2.5D MAPPING PIPELINE EXECUTION SUMMARY ===")
print(df_summary[cols_display].to_string(index=False))
print("=" * 60 + "\n")

# ------------------------------------------------------------------------------
# 6. Assertions & Verification
# ------------------------------------------------------------------------------
# 1. All synthetic regions pass validation
for region in synthetic_regions:
    assert validate_region_features(region) is True

# 2. Pedestrian importance > Road importance
assert pedestrian_imp > road_imp, (
    f"Pedestrian importance ({pedestrian_imp}) must exceed Road ({road_imp})"
)

# 3. Pedestrian resolution is finer than Road resolution (smaller meters = finer)
assert pedestrian_res < road_res, (
    f"Pedestrian resolution ({pedestrian_res}m) must be finer than Road ({road_res}m)"
)

# 4. Confidence and uncertainty are distinct fields
for r in synthetic_regions:
    assert r.confidence != r.uncertainty, (
        f"Region {r.region_id} has matching confidence and uncertainty. They must be distinct."
    )

print("We have a stable common RegionFeatures interface. Synthetic data can enter through this interface and reach the existing Importance → Resolution → 2.5D mapping pipeline without changing the core algorithm.")


=== ADAPTIVE 2.5D MAPPING PIPELINE EXECUTION SUMMARY ===
 region_id semantic_label distance  importance_score resolution  confidence  uncertainty
       101     Pedestrian    70.0m            0.7185      0.10m        0.88         0.12
       102           Road    70.0m            0.1155      2.00m        0.96         0.04
       103            Car    20.0m            0.6860      0.50m        0.92         0.10
       104     Vegetation    15.0m            0.4325      0.50m        0.70         0.75
       105       Building    95.0m            0.1900      2.00m        0.85         0.25

We have a stable common RegionFeatures interface. Synthetic data can enter through this interface and reach the existing Importance → Resolution → 2.5D mapping pipeline without changing the core algorithm.
